# 평점 통계분석 기반 EDA
> 무신사 3개 브랜드 × 3개 카테고리 리뷰 데이터

## 0. Import & 환경 설정

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family']      = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 로드

In [ ]:
cr = pd.read_csv("cleaned_reviews.csv")
cp = pd.read_csv("cleaned_products.csv")

print(f"reviews  shape : {cr.shape}")
print(f"products shape : {cp.shape}")
display(cr.head(2))

## 2. 리뷰 × 상품 병합 (goodsNo 기준)

In [ ]:
merged_df = pd.merge(cr, cp, on='goodsNo', how='inner')
print(f"병합 후 shape : {merged_df.shape}")

---
## 3. 전체 평점 분포

In [ ]:
rating_counts = cr['평점'].value_counts().sort_index()
print(cr['평점'].describe())
print(f"\nNaN(무효평점): {cr['평점'].isna().sum()}건")
print("\n평점별 건수:")
print(rating_counts)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 히스토그램 ──────────────────────────────────────────────
sns.histplot(cr['평점'].dropna(), bins=5, kde=False, color='skyblue', ax=axes[0])
axes[0].set_title('평점 빈도 분포 (Histogram)', fontsize=13)
axes[0].set_xlabel('평점'); axes[0].set_ylabel('리뷰 개수')

# ── 막대 그래프 ─────────────────────────────────────────────
ax = sns.barplot(x=rating_counts.index, y=rating_counts.values, palette='viridis', ax=axes[1])
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', xytext=(0, 5), textcoords='offset points', fontsize=10)
axes[1].set_title('평점별 리뷰 개수', fontsize=13)
axes[1].set_xlabel('평점'); axes[1].set_ylabel('리뷰 개수')

plt.tight_layout()
plt.show()

###  결과 해석
- 전체 리뷰의 **5점 비율이 압도적으로 높음** → 무신사 리뷰 특성상 긍정 편향(Positivity Bias)이 강하게 존재
- 평균 평점만으로는 브랜드 간 차이를 구분하기 어려움 → **5점 비율, 1~2점 비율 등 분포 분석이 필수**
- 1~2점 저평점 리뷰 수는 적지만, 부정 리뷰 1건이 전체 평점에 미치는 영향이 리뷰 수가 적은 상품일수록 크게 작용함

---
## 4. 브랜드별 평점 통계 및 신뢰구간 분석

In [ ]:
brand_order = ['제멋', '필루미네이트', '트래블']

stats = merged_df.groupby('브랜드')['평점'].agg(['count', 'mean', 'std']).reindex(brand_order)
stats.columns = ['리뷰수', '평균평점', '표준편차']
stats['표준오차'] = stats['표준편차'] / np.sqrt(stats['리뷰수'])
stats['CI_상한']  = stats['평균평점'] + 1.96 * stats['표준오차']
stats['CI_하한']  = stats['평균평점'] - 1.96 * stats['표준오차']
stats['CI_폭']    = stats['CI_상한'] - stats['CI_하한']

display(stats)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for i, brand in enumerate(stats.index):
    row   = stats.loc[brand]
    width = row['CI_폭']

    ax.plot(i, width, 'o', color='royalblue', markersize=12)
    ax.errorbar(i, width, yerr=width * 0.15, capsize=10, color='royalblue', linewidth=2.5)

    ax.text(i + 0.08, width,
            f"폭: {width:.5f}", va='center', fontweight='bold', fontsize=12)
    ax.text(i, width + width * 0.15 + 0.0001,
            f"Upper: {row['CI_상한']:.4f}", ha='center', va='bottom',
            color='red', fontweight='bold', fontsize=10)
    ax.text(i, width - width * 0.15 - 0.0001,
            f"Lower: {row['CI_하한']:.4f}", ha='center', va='top',
            color='blue', fontweight='bold', fontsize=10)

ax.set_xticks(range(len(brand_order)))
ax.set_xticklabels(brand_order, fontsize=12)
ax.set_title('브랜드별 평점 변동성(폭) 및 실제 신뢰구간 범위', fontsize=15, pad=20)
ax.set_ylabel('신뢰구간 폭 (CI Width)', fontsize=12)
ax.set_xlabel('브랜드', fontsize=12)
ax.set_ylim(0, stats['CI_폭'].max() * 1.6)
ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

### 결과 해석
- **CI 폭이 좁다 = 평균 평점 추정의 통계적 불확실성이 작다** (표본 수가 많을수록 자연스럽게 좁아짐)
  - 제멋(폭 가장 좁음) > 트래블 > 필루미네이트 순으로 통계적 안정성 높음
  - 단, 이는 **"브랜드 만족도가 높다"는 직접 근거가 아님** — 리뷰 수가 많아서 추정 오차가 줄어든 것
- **실제 만족도 평가는** 평균 평점 단독이 아니라 **부정 리뷰 비율 + 토픽별 부정 신호를 함께 고려**해야 함
- 신뢰구간 상/하한값 자체(4.77~4.78 수준)는 세 브랜드 모두 비슷하여, 다음 분석에서 **분포(5점·저평점 비율)**로 연결 필요

---
## 5. 브랜드 × 카테고리별 평점 분석

In [ ]:
brand_cat_stat = merged_df.groupby(['브랜드', '카테고리'])['평점'].describe()
display(brand_cat_stat)

### 결과 해석
- 브랜드×카테고리 간 **평균 평점 차이는 0.01~0.04 수준으로 매우 작음** → 평균만으로 순위 매기는 것은 무의미
- 평균이 같아도 **분포(5점 비율, 1~2점 비율)는 크게 다를 수 있음** → 아래 분포 분석으로 연결

### 5-1. 평점 구간 분포 — Stacked Bar

In [ ]:
score_dist = (
    merged_df.groupby(['브랜드', '카테고리', '평점'])
    .size().reset_index(name='count')
)
total_cnt = merged_df.groupby(['브랜드', '카테고리'])['평점'].count().reset_index(name='total')
score_dist = score_dist.merge(total_cnt, on=['브랜드', '카테고리'])
score_dist['비율'] = score_dist['count'] / score_dist['total']

pivot = score_dist.pivot_table(
    index=['브랜드', '카테고리'], columns='평점', values='비율', fill_value=0
)
pivot.columns = [f'{int(c)}점' for c in pivot.columns]
pivot.index   = [f"{b}/{c}" for b, c in pivot.index]

colors = ['#d73027', '#f46d43', '#fee08b', '#a6d96a', '#1a9850']
pivot.plot(kind='bar', stacked=True, figsize=(14, 6), color=colors)
plt.title('브랜드×카테고리별 평점 구간 분포 (Stacked Bar)', fontsize=14)
plt.xlabel('브랜드 / 카테고리')
plt.ylabel('비율')
plt.xticks(rotation=40, ha='right')
plt.legend(title='평점', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 결과 해석
- **5점(초록) 비율이 전 구간에서 지배적** — 무신사 플랫폼 특성상 긍정 편향이 강함
- 평균은 비슷하지만 **빨간 영역(1~2점)의 두께가 카테고리별로 차이** → 단순 평균 비교의 한계를 보여줌
- 특히 리뷰 수가 적은 카테고리는 저평점 1건이 stacked bar에서 눈에 띄게 나타남 → 리스크 지수 분석과 연결

### 5-2. 5점 비율 및 1~2점 비율 비교

In [ ]:
rating_agg = merged_df.groupby(['브랜드', '카테고리'])['평점'].agg(
    total_count = 'count',
    count_5     = lambda x: (x == 5).sum(),
    count_low   = lambda x: (x <= 2).sum(),
).reset_index()

rating_agg['ratio_5']   = rating_agg['count_5']   / rating_agg['total_count']
rating_agg['ratio_low'] = rating_agg['count_low']  / rating_agg['total_count']
rating_agg['label']     = rating_agg['브랜드'] + ' / ' + rating_agg['카테고리']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── 5점 비율 ────────────────────────────────────────────────
bars0 = sns.barplot(ax=axes[0], data=rating_agg.sort_values('ratio_5'),
                    x='label', y='ratio_5', palette='Blues_d')
for p in bars0.patches:
    bars0.annotate(f"{p.get_height():.3f}",
                   (p.get_x() + p.get_width()/2., p.get_height()),
                   ha='center', va='bottom', xytext=(0,4), textcoords='offset points', fontsize=10)
axes[0].set_title('브랜드×카테고리별 5점 비율', fontsize=13)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=40, ha='right')
axes[0].set_ylabel('5점 비율'); axes[0].set_ylim(0, 1.05)

# ── 1~2점 비율 ──────────────────────────────────────────────
bars1 = sns.barplot(ax=axes[1], data=rating_agg.sort_values('ratio_low', ascending=False),
                    x='label', y='ratio_low', palette='Reds_d')
for p in bars1.patches:
    bars1.annotate(f"{p.get_height():.4f}",
                   (p.get_x() + p.get_width()/2., p.get_height()),
                   ha='center', va='bottom', xytext=(0,4), textcoords='offset points', fontsize=10)
axes[1].set_title('브랜드×카테고리별 1~2점 비율', fontsize=13)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=40, ha='right')
axes[1].set_ylabel('1~2점 비율')

plt.suptitle('평균이 비슷해도 분포는 다르다 — 5점 vs 1~2점 비율 비교', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

display(rating_agg[['브랜드','카테고리','total_count','ratio_5','ratio_low']]
        .sort_values('ratio_low', ascending=False).reset_index(drop=True))

### 결과 해석
- **5점 비율**: 리뷰수가 많은 브랜드(제멋)가 상대적으로 안정적이나, 카테고리별 편차 존재
- **1~2점 비율**: 절댓값은 작지만(1~3% 수준), 이 비율이 높은 카테고리는 **소비자 불만이 반복적으로 발생**하는 구조적 이슈 가능성
- 리뷰 수가 적은 카테고리일수록 1~2점 비율 추정이 불안정 → **리스크 지수 분석에서 리뷰수(n)를 함께 고려해야 하는 이유**

---
## 6. 품질 리스크 지수 (Risk Index)

**취약도 지표 정의**
$$\text{risk\_score} = \frac{1 - \text{ratio\_5}}{n}$$

| 항목 | 의미 |
|---|---|
| `ratio_5` | 5점 비율 — 높을수록 만족도 높음 |
| `n` | 전체 리뷰 수 — 많을수록 추정 안정 |
| `risk_score` | 리뷰 수가 적고, 5점 비율이 낮을수록 높아지는 **취약도** |
| `risk_index` | risk_score × 1,000,000 (가독성 스케일) |

> 리뷰가 매우 적은 그룹(n이 작은 경우)은 risk_index가 과대 산출될 수 있으므로, **반드시 n과 함께 해석**해야 함.

In [ ]:
risk_df = merged_df.groupby(['브랜드', '카테고리'])['평점'].agg(
    total_count = 'count',
    mean_rating = 'mean',
    count_5     = lambda x: (x == 5).sum(),
).reset_index()

risk_df['ratio_5']    = risk_df['count_5'] / risk_df['total_count']

# 취약도 지표: (1 - ratio_5) / n
risk_df['risk_score'] = (1 - risk_df['ratio_5']) / risk_df['total_count']
risk_df['risk_index'] = risk_df['risk_score'] * 1_000_000

priority_list = risk_df.sort_values('risk_index', ascending=False).reset_index(drop=True)

display(priority_list[['브랜드','카테고리','total_count','mean_rating','ratio_5','risk_index']])

In [ ]:
brands = ['제멋', '트래블', '필루미네이트']
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

for i, brand in enumerate(brands):
    bdata = (priority_list[priority_list['브랜드'] == brand]
             .sort_values('risk_index', ascending=False))

    bars = sns.barplot(ax=axes[i], x='카테고리', y='risk_index',
                       data=bdata, palette='Reds_r')

    # 막대 위 수치 + 리뷰수(n) 함께 표시
    for j, p in enumerate(bars.patches):
        n_val = bdata.iloc[j]['total_count']
        axes[i].annotate(
            f"{p.get_height():.2f}\n(n={int(n_val):,})",
            (p.get_x() + p.get_width()/2., p.get_height()),
            ha='center', va='bottom', fontsize=10, fontweight='bold',
            xytext=(0, 5), textcoords='offset points'
        )

    axes[i].set_title(f'[{brand}] 카테고리별 리스크 지수', fontsize=13, pad=12)
    axes[i].set_ylabel('Risk Index')
    axes[i].set_xlabel('카테고리')

plt.suptitle('브랜드별 카테고리 안정성(리스크) 정밀 비교  |  막대 위 n = 리뷰 수',
             fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

### 결과 해석
- **risk_index가 높다 = 부정 평점 1개의 영향력이 크고 구조적으로 취약한 카테고리**
- **3개 브랜드 중 필루미네이트 아우터의 리스크가 가장 높음**
  - 다른 카테고리 대비 약 11배 위험 수준 (n이 작고 5점 비율도 낮음)
- **제멋 → 아우터** 카테고리 상품 관리 우선 필요
- **트래블 → 하의** 카테고리 상품 관리 우선 필요
- **필루미네이트 → 아우터** 상품 관리 최우선 (다른 카테고리보다 약 11배 위험)

> n이 작은 카테고리(필루미네이트 아우터 등)는 risk_index가 과대 산출될 수 있음  
> 위기 신호 포착이 목적이라면 **평균 평점보다 최근 부정 리뷰 비율 변화 추이**를 함께 모니터링하는 것이 더 중요